In [2]:
import re

class Expr:
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args else []

    def __repr__(self):
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(self.args)})"

    def __eq__(self, other):
        return isinstance(other, Expr) and self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, tuple(self.args)))

class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        premises_str = " ^ ".join(map(str, self.premises))
        return f"{premises_str} => {self.conclusion}"

def is_variable(x):
    return isinstance(x, str) and len(x) > 0 and (x.islower() or "_" in x)

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is False:
        return False
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, Expr) and isinstance(y, Expr):
        if x.name != y.name or len(x.args) != len(y.args):
            return False
        for ax, ay in zip(x.args, y.args):
            theta = unify(ax, ay, theta)
        return theta
    else:
        return False

def unify_var(var, val, theta):
    if var in theta:
        return unify(theta[var], val, theta)
    elif val in theta:
        return unify(var, theta[val], theta)
    else:
        theta[var] = val
        return theta

def subst(theta, expr):
    if isinstance(expr, str):
        return theta.get(expr, expr)
    return Expr(expr.name, [subst(theta, arg) for arg in expr.args])

var_counter = 0

def standardize_variables(rule):
    global var_counter
    var_counter += 1
    theta = {}

    def collect_vars(expr):
        return {arg for arg in expr.args if is_variable(arg)}

    all_vars = set()
    for p in rule.premises:
        all_vars.update(collect_vars(p))
    all_vars.update(collect_vars(rule.conclusion))

    for var in all_vars:
        theta[var] = f"{var}_{var_counter}"

    new_premises = [subst(theta, p) for p in rule.premises]
    new_conclusion = subst(theta, rule.conclusion)
    return Rule(new_premises, new_conclusion)

# =====================================================================
# FLEXIBLE USER STRING PARSER ENGINE
# =====================================================================

def parse_expr(s):
    s = s.strip()
    if not s:
        raise ValueError("Expression target segment cannot be empty")

    match = re.match(r"^([A-Za-z0-9_]+)\((.*)\)$", s)
    if match:
        name = match.group(1)
        args_str = match.group(2).strip()
        args = [arg.strip() for arg in args_str.split(",") if arg.strip()] if args_str else []
        return Expr(name, args)

    if re.match(r"^[A-Za-z0-9_]+$", s):
        return Expr(s, [])

    raise ValueError(f"Could not cleanly resolve predicate formatting on: '{s}'")

def parse_rule(s):
    s = s.strip()
    s_norm = s.replace("->", "=>").replace("THEN", "=>").replace("then", "=>")

    if "=>" not in s_norm:
        raise ValueError("Missing implication connector assignment platform (e.g. use '=>', '->')")

    left, right = s_norm.split("=>", 1)
    conclusion = parse_expr(right)

    left_norm = left.replace("AND", "^").replace("and", "^").replace("+", "^").replace(",", "^")
    left_norm = left_norm.replace("IF", "").replace("if", "")

    premises = []
    for part in left_norm.split("^"):
        if part.strip():
            premises.append(parse_expr(part))

    if not premises:
        raise ValueError("Rules structural format profiles require at least one base premise condition")

    return Rule(premises, conclusion)

# =====================================================================
# FORWARD CHAINING CORE LOGIC LOOP (FOL-FC-ASK)
# =====================================================================

def fol_fc_ask(kb_rules, kb_facts, alpha):
    facts = set(kb_facts)
    iteration = 1

    print("\nExecuting Forward Chaining Inference Engine...")

    while True:
        new_facts = set()

        for rule in kb_rules:
            std_rule = standardize_variables(rule)
            substitutions = [{}]

            for premise in std_rule.premises:
                next_subs = []
                for theta in substitutions:
                    for fact in facts:
                        subst_premise = subst(theta, premise)
                        t = unify(subst_premise, fact, theta.copy())
                        if t is not False:
                            next_subs.append(t)
                substitutions = next_subs

            for theta in substitutions:
                q_prime = subst(theta, std_rule.conclusion)

                if not any(unify(q_prime, f) is not False for f in facts.union(new_facts)):
                    new_facts.add(q_prime)
                    print(f"  [Derived]: {q_prime} from {rule}")

                    phi = unify(q_prime, alpha)
                    if phi is not False:
                        return phi

        if not new_facts:
            break

        facts.update(new_facts)
        iteration += 1

    return False

# =====================================================================
# CONSOLE RUNTIME CONTROLLER
# =====================================================================

if __name__ == "__main__":
    print("=======================================================")
    print("FIRST-ORDER LOGIC REASONING PROGRAM")
    print("=======================================================")
    print("Supported syntax template models:")
    print("  Rules: Fever(x) AND SoreThroat(x) THEN SignsOfInfection(x)")
    print("  Facts: Fever(John)")
    print("-------------------------------------------------------\n")

    user_rules = []
    user_facts = []

    # 1. Capture Dynamic Rules List
    print("Enter rules configuration mappings (type 'done' to stop):")
    while True:
        rule_in = input("  Add Rule: ").strip()
        if rule_in.lower() == 'done':
            break
        if not rule_in:
            continue
        try:
            parsed_rule = parse_rule(rule_in)
            user_rules.append(parsed_rule)
            print(f"    Saved: {parsed_rule}")
        except Exception as err:
            print(f"    [Error]: {err}. Re-check layout string structure.")

    # 2. Capture Baseline Facts Matrix
    print("\nEnter initial grounding axioms (type 'done' to stop):")
    while True:
        fact_in = input("  Add Fact: ").strip()
        if fact_in.lower() == 'done':
            break
        if not fact_in:
            continue
        try:
            parsed_fact = parse_expr(fact_in)
            user_facts.append(parsed_fact)
            print(f"    Saved: {parsed_fact}")
        except Exception as err:
            print(f"    [Error]: {err}. Re-check layout string structure.")

    # 3. Target Query Definition
    print("")
    while True:
        goal_in = input("Enter target query goal declaration (Alpha): ").strip()
        if goal_in:
            try:
                target_goal = parse_expr(goal_in)
                break
            except Exception as err:
                print(f"    [Error]: {err}. Try typing again.")

    # 4. Trigger Execution Pipeline
    if not user_rules or not user_facts:
        print("\nProcess ended: Knowledge Base state profiles cannot be empty.")
    else:
        substitution_map = fol_fc_ask(user_rules, user_facts, target_goal)

        if substitution_map is not False:
            print(f"\nGoal Match Successful. Final Substitution Profiles (Theta): {substitution_map}")
        else:
            print("\nTarget goal declaration could not be inferred from the given knowledge state rules database.")


FIRST-ORDER LOGIC REASONING PROGRAM
Supported syntax template models:
  Rules: Fever(x) AND SoreThroat(x) THEN SignsOfInfection(x)
  Facts: Fever(John)
-------------------------------------------------------

Enter rules configuration mappings (type 'done' to stop):
  Add Rule: IF Fever(x) AND SoreThroat(x) THEN SignsOfInfection(x)
    Saved: Fever(x) ^ SoreThroat(x) => SignsOfInfection(x)
  Add Rule: IF SignsOfInfection(x) AND MuscleAches(x) THEN TestForFlu(x)
    Saved: SignsOfInfection(x) ^ MuscleAches(x) => TestForFlu(x)
  Add Rule:  done

Enter initial grounding axioms (type 'done' to stop):
  Add Fact:  Fever(John)
    Saved: Fever(John)
  Add Fact: SoreThroat(John)
    Saved: SoreThroat(John)
  Add Fact: MuscleAches(John)
    Saved: MuscleAches(John)
  Add Fact: done

Enter target query goal declaration (Alpha): TestForFlu(John)

Executing Forward Chaining Inference Engine...
  [Derived]: SignsOfInfection(John) from Fever(x) ^ SoreThroat(x) => SignsOfInfection(x)
  [Derived]: Te